# Axioms and notation — admissibility witness

**Formal source:** [`../00_axioms_and_notation.md`](../00_axioms_and_notation.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rho_min = 1e-3
raw = np.array([-2.0, 0.0, 2.0])
rho = rho_min + np.maximum(raw, 0) + np.log1p(np.exp(-np.abs(raw)))
low = np.array([0.0, 10.0, 20.0])
high = np.array([10.0, 20.0, 30.0])
nu = np.array([-3.0, 0.0, 3.0])
omega = low + (high - low) / (1 + np.exp(-nu))
assert np.all(rho > 0)
assert np.all((omega >= low) & (omega <= high))
operator = np.diag([2.0, 0.1])
gramian = operator.T @ operator
values, vectors = np.linalg.eigh(gramian)
projector = vectors[:, values >= 1] @ vectors[:, values >= 1].T
np.testing.assert_allclose(projector, np.diag([1.0, 0.0]))
try:
    np.linalg.cholesky(np.diag([1.0, -1.0]))
    raise AssertionError("indefinite covariance must be rejected")
except np.linalg.LinAlgError:
    pass
print({"rho": rho.tolist(), "omega": omega.tolist(), "rank": int(round(np.trace(projector)))})

In [ ]:
print('THEORY_DEMO_PASS::00_axioms_and_notation')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')